# VoyageRL 实验笔记
从项目根目录启动 Jupyter。数据示例使用本地归档路径；真实 API 调用前请配置 `.env`。训练日志由你自己的训练生成。

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), Path.cwd().parent)
     if (p / "run_tool_loop_infer.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter from the project root or notebooks directory.")
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")


# Question expansion: 50 new examples

Reads `data/train_seed.jsonl` and generates at most 50 deduplicated questions.
Configure `EXPANSION_API_KEY`, `EXPANSION_BASE_URL`, and `EXPANSION_MODEL` in the root `.env`.
This notebook calls the configured remote model and may incur API charges.
The script bounds API attempts and retains partial output on failure.
Each run replaces `data/train_expansion_50.jsonl`.


In [ ]:
from importlib.metadata import version

print(f"openai version: {version('openai')}")


In [ ]:
from pathlib import Path
import importlib.util

# 支持从本目录或上一级项目目录启动 Jupyter。
project_dir = next(
    (directory for directory in (Path.cwd(), Path.cwd().parent)
     if (directory / "test_expand_50.py").is_file()),
    None,
)
if project_dir is None:
    raise FileNotFoundError("请从项目根目录或 notebooks 目录启动此 notebook。")
spec = importlib.util.spec_from_file_location("expansion_test", project_dir / "test_expand_50.py")
expansion_test = importlib.util.module_from_spec(spec)
spec.loader.exec_module(expansion_test)
print(f"目标新增条数：{expansion_test.MAX_NEW_SAMPLES}")
print(f"模型：{expansion_test.MODEL}")
print(f"输出文件：{expansion_test.OUTPUT_PATH}")


## 开始 50 条测试

运行下一个单元格即开始真实 API 调用。通常每次生成 5 条，约 10 次请求达到 50 条；若有重复或失败，受上述请求上限限制。达到 50 条后单元格结束，Jupyter 内核仍可正常使用。


In [ ]:
exit_code = expansion_test.main()
if exit_code != 0:
    raise RuntimeError(f"扩充测试未完成，退出码 {exit_code}；详见上方日志。")


In [ ]:
import json

output_path = expansion_test.OUTPUT_PATH
if output_path.exists():
    rows = [json.loads(line) for line in output_path.read_text(encoding="utf-8").splitlines()]
    print(f"测试输出共 {len(rows)} 条，展示前 5 条：")
    for item in rows[:5]:
        print(item["question"])
